# Impact Analysis: SDK, Retry Mechanism & System Prompt

**Goal:** Automatically measure the individual and combined impact of three factors on LLM agent reliability:
1. **SDK** — `@ai-sdk/anthropic` vs `@ai-sdk/openai-compatible`
2. **Retry mechanism** — `max_retries=0` (off) vs `max_retries=2` (on)
3. **System prompt** — 6 variants (anthropic, beast, codex, gemini, qwen, trinity)

**Key metrics:** `success_rate` (has final response), `empty_avg` (empty responses per run), `error_rate` (runs with errors), `api_calls_avg`

In [8]:
import pandas as pd
from IPython.display import display, Markdown

df = pd.read_csv("results.csv")

FACTORS = ["sdk", "max_retries", "system_prompt_name"]
METRICS = {
    "n":            ("run_id", "count"),
    "success_rate": ("has_final_response", "mean"),
    "empty_avg":    ("empty", "mean"),
    "error_rate":   ("errors", lambda s: (s > 0).mean()),
    "api_calls_avg":("api_calls", "mean"),
}
RANK_SORT = ["success_rate", "empty_avg", "error_rate", "api_calls_avg", "n"]
RANK_ASC  = [False,          True,        True,         True,            False]

def agg(keys):
    t = df.groupby(keys, dropna=False).agg(**METRICS).reset_index()
    for c in ["success_rate", "empty_avg", "error_rate", "api_calls_avg"]:
        t[c] = t[c].round(3)
    return t

def rank(t, group_by_sdk=False):
    if group_by_sdk and "sdk" in t.columns:
        return t.sort_values(["sdk"] + RANK_SORT, ascending=[True] + RANK_ASC).reset_index(drop=True).rename_axis("rank")
    return t.sort_values(RANK_SORT, ascending=RANK_ASC).reset_index(drop=True).rename_axis("rank")

In [9]:
print(f"{len(df)} runs ")
print(f"SDKs: {sorted(df['sdk'].unique())}")
print(f"retries: {sorted(df['max_retries'].unique())}")
print(f"prompts: {sorted(df['system_prompt_name'].unique())}")

600 runs 
SDKs: ['@ai-sdk/anthropic', '@ai-sdk/openai-compatible']
retries: [0, 2]
prompts: ['anthropic', 'beast', 'codex', 'gemini', 'qwen', 'trinity']


## 1. SDK impact

In [10]:
display(rank(agg(["sdk"])))

,sdk,n,success_rate,empty_avg,error_rate,api_calls_avg
rank,,,,,,
0,@ai-sdk/anthropic,300,0.92,0.133,0.060,10.78
1,@ai-sdk/openai-compatible,300,0.34,1.587,0.087,5.74


## 2. SDK + Retry mechanism impact

In [11]:
display(Markdown("**Per SDK (retry x sdk interaction)**"))
retry_by_sdk = rank(agg(["sdk", "max_retries"]), group_by_sdk=True)
display(retry_by_sdk)

**Per SDK (retry x sdk interaction)**

,sdk,max_retries,n,success_rate,empty_avg,error_rate,api_calls_avg
rank,,,,,,,
0,@ai-sdk/anthropic,2,150,0.927,0.100,0.033,10.440
1,@ai-sdk/anthropic,0,150,0.913,0.167,0.087,11.120
2,@ai-sdk/openai-compatible,2,150,0.520,2.213,0.087,7.833
3,@ai-sdk/openai-compatible,0,150,0.160,0.960,0.087,3.647


## 3. System prompt impact (marginal)

In [12]:
display(Markdown("**Global (marginal)**"))
display(rank(agg(["system_prompt_name"])))

**Global (marginal)**

,system_prompt_name,n,success_rate,empty_avg,error_rate,api_calls_avg
rank,,,,,,
0,anthropic,100,0.67,0.76,0.01,9.66
1,qwen,100,0.67,0.95,0.01,7.21
2,trinity,100,0.65,0.67,0.06,8.60
3,gemini,100,0.64,0.90,0.16,6.11
4,beast,100,0.62,0.92,0.06,8.91
5,codex,100,0.53,0.96,0.14,9.07


## 4. Full grid (SDK + Retry + System prompt)

In [13]:
display(Markdown("**Per SDK (prompt x sdk interaction)**"))
display(rank(agg(["sdk", "max_retries", "system_prompt_name"]), group_by_sdk=True))

**Per SDK (prompt x sdk interaction)**

,sdk,max_retries,system_prompt_name,n,success_rate,empty_avg,error_rate,api_calls_avg
rank,,,,,,,,
0,@ai-sdk/anthropic,0,trinity,25,1.00,0.08,0.04,9.76
1,@ai-sdk/anthropic,2,gemini,25,0.96,0.04,0.00,4.92
2,@ai-sdk/anthropic,2,beast,25,0.96,0.04,0.00,12.64
3,@ai-sdk/anthropic,2,anthropic,25,0.96,0.16,0.00,12.48
4,@ai-sdk/anthropic,2,qwen,25,0.92,0.08,0.04,8.24
5,@ai-sdk/anthropic,2,codex,25,0.92,0.08,0.08,10.88
6,@ai-sdk/anthropic,0,anthropic,25,0.92,0.12,0.00,12.88
7,@ai-sdk/anthropic,0,beast,25,0.92,0.12,0.12,12.40
8,@ai-sdk/anthropic,0,qwen,25,0.92,0.20,0.00,9.00


## 5. Verdict

In [14]:
print("Best SDK:")
display(rank(agg(["sdk"])).head(1))

for sdk in sorted(df["sdk"].unique()):
    for retries in sorted(df["max_retries"].unique()):
        print(f"\nTop 3 system prompts — sdk={sdk}, max_retries={retries}:")
        sub = df[(df["sdk"] == sdk) & (df["max_retries"] == retries)]
        t = sub.groupby("system_prompt_name", dropna=False).agg(**METRICS).reset_index()
        for c in ["success_rate", "empty_avg", "error_rate", "api_calls_avg"]:
            t[c] = t[c].round(3)
        display(t.sort_values(RANK_SORT, ascending=RANK_ASC).reset_index(drop=True).head(3))

Best SDK:


,sdk,n,success_rate,empty_avg,error_rate,api_calls_avg
rank,,,,,,
0,@ai-sdk/anthropic,300,0.92,0.133,0.06,10.78



Top 3 system prompts — sdk=@ai-sdk/anthropic, max_retries=0:


,system_prompt_name,n,success_rate,empty_avg,error_rate,api_calls_avg
0,trinity,25,1.00,0.08,0.04,9.76
1,anthropic,25,0.92,0.12,0.00,12.88
2,beast,25,0.92,0.12,0.12,12.40



Top 3 system prompts — sdk=@ai-sdk/anthropic, max_retries=2:


,system_prompt_name,n,success_rate,empty_avg,error_rate,api_calls_avg
0,gemini,25,0.96,0.04,0.0,4.92
1,beast,25,0.96,0.04,0.0,12.64
2,anthropic,25,0.96,0.16,0.0,12.48



Top 3 system prompts — sdk=@ai-sdk/openai-compatible, max_retries=0:


,system_prompt_name,n,success_rate,empty_avg,error_rate,api_calls_avg
0,trinity,25,0.24,0.80,0.04,3.72
1,qwen,25,0.20,0.92,0.00,3.80
2,anthropic,25,0.20,1.00,0.00,3.64



Top 3 system prompts — sdk=@ai-sdk/openai-compatible, max_retries=2:


,system_prompt_name,n,success_rate,empty_avg,error_rate,api_calls_avg
0,gemini,25,0.64,2.08,0.00,7.04
1,qwen,25,0.64,2.60,0.00,7.80
2,anthropic,25,0.60,1.76,0.04,9.64


## History 



PROXY_PORT=8027 OPENCODE_EMPTY_RESPONSE_RETRIES=0 OPENCODE_SYSTEM_PROMPT=qwen SDK=@ai-sdk/openai-compatible N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_SYSTEM_PROMPT=qwen SDK=@ai-sdk/openai-compatible N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_EMPTY_RESPONSE_RETRIES=0 OPENCODE_SYSTEM_PROMPT=qwen SDK=@ai-sdk/anthropic N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_SYSTEM_PROMPT=qwen SDK=@ai-sdk/anthropic N_RUNS=13 ./test_retry/run_test.sh

PROXY_PORT=8027 OPENCODE_EMPTY_RESPONSE_RETRIES=0 OPENCODE_SYSTEM_PROMPT=beast SDK=@ai-sdk/openai-compatible N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=9022 OPENCODE_SYSTEM_PROMPT=beast SDK=@ai-sdk/openai-compatible N_RUNS=5 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_EMPTY_RESPONSE_RETRIES=0 OPENCODE_SYSTEM_PROMPT=beast SDK=@ai-sdk/anthropic N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_SYSTEM_PROMPT=beast SDK=@ai-sdk/anthropic N_RUNS=13 ./test_retry/run_test.sh

PROXY_PORT=8027 OPENCODE_SYSTEM_PROMPT=anthropic SDK=@ai-sdk/anthropic N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_SYSTEM_PROMPT=anthropic SDK=@ai-sdk/openai-compatible N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_EMPTY_RESPONSE_RETRIES=0 OPENCODE_SYSTEM_PROMPT=anthropic SDK=@ai-sdk/openai-compatible N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_EMPTY_RESPONSE_RETRIES=0 OPENCODE_SYSTEM_PROMPT=anthropic SDK=@ai-sdk/anthropic N_RUNS=13 ./test_retry/run_test.sh

PROXY_PORT=8027 OPENCODE_SYSTEM_PROMPT=gemini SDK=@ai-sdk/anthropic N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_SYSTEM_PROMPT=gemini SDK=@ai-sdk/openai-compatible N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_EMPTY_RESPONSE_RETRIES=0 OPENCODE_SYSTEM_PROMPT=gemini SDK=@ai-sdk/anthropic N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_EMPTY_RESPONSE_RETRIES=0 OPENCODE_SYSTEM_PROMPT=gemini SDK=@ai-sdk/openai-compatible N_RUNS=13 ./test_retry/run_test.sh

PROXY_PORT=8027 OPENCODE_SYSTEM_PROMPT=trinity SDK=@ai-sdk/anthropic N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_SYSTEM_PROMPT=trinity SDK=@ai-sdk/openai-compatible N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_EMPTY_RESPONSE_RETRIES=0 OPENCODE_SYSTEM_PROMPT=trinity SDK=@ai-sdk/anthropic N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8027 OPENCODE_EMPTY_RESPONSE_RETRIES=0 OPENCODE_SYSTEM_PROMPT=trinity SDK=@ai-sdk/openai-compatible N_RUNS=13 ./test_retry/run_test.sh

PROXY_PORT=8022 OPENCODE_SYSTEM_PROMPT=codex SDK=@ai-sdk/anthropic N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=8021 OPENCODE_SYSTEM_PROMPT=codex SDK=@ai-sdk/openai-compatible N_RUNS=13 ./test_retry/run_test.sh
PROXY_PORT=7777 OPENCODE_EMPTY_RESPONSE_RETRIES=0 OPENCODE_SYSTEM_PROMPT=codex SDK=@ai-sdk/anthropic N_RUNS=4 ./test_retry/run_test.sh
PROXY_PORT=8020 OPENCODE_EMPTY_RESPONSE_RETRIES=0 OPENCODE_SYSTEM_PROMPT=codex SDK=@ai-sdk/openai-compatible N_RUNS=13 ./test_retry/run_test.sh
